In [7]:
import sys
sys.path.insert(0, '/Users/vahid/Downloads/FERNN-master/moving_mnist_fp')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Import Moving MNIST dataset
from moving_mnist_dataset import MovingMNISTDataset

# Force reload of models module
import importlib
import moving_mnist_models
importlib.reload(moving_mnist_models)

print("✓ All imports successful")

✓ All imports successful


Output shape: torch.Size([2, 10, 1, 28, 28])
Velocity probs shape: torch.Size([2, 20, 25])
Velocity probs sum (first batch, first step): 1.0
Loss: 0.001270188600756228
Grad OK: cell.conv_h.weight | norm=0.000031
Gradient flow: True
Sanity check complete.


In [11]:
# Sanity check: Forward + backward on synthetic data

def generate_moving_dot(batch_size=2, seq_len=20, H=28, W=28, speed=1):
    """Generate simple synthetic moving dot sequence"""
    seq = torch.zeros(batch_size, seq_len, 1, H, W)
    for b in range(batch_size):
        x = np.random.randint(0, W)
        y = np.random.randint(0, H)
        vx = np.random.choice([-speed, 0, speed])
        vy = np.random.choice([-speed, 0, speed])
        for t in range(seq_len):
            seq[b, t, 0, y % H, x % W] = 1.0
            x += vx
            y += vy
    return seq

# Generate data
batch_size = 2
seq_len = 20
input_frames = 10
pred_len = seq_len - input_frames

seq = generate_moving_dot(batch_size=batch_size, seq_len=seq_len)
input_seq = seq[:, :input_frames]
target_seq = seq[:, input_frames:]

print(f"✓ Input shape: {input_seq.shape}, Target shape: {target_seq.shape}")

# Initialize model
from moving_mnist_models import Seq2SeqFERNN

model = Seq2SeqFERNN(
    input_channels=1,
    hidden_channels=16,
    height=28,
    width=28,
    v_range=2,
    decoder_conv_layers=1,
    pool_type='max'
)
model.train()

# Forward pass
try:
    output, vel_probs = model(
        input_seq,
        pred_len=pred_len,
        teacher_forcing_ratio=0.0,
        target_seq=target_seq,
        return_vel_probs=True
    )
    print(f"✓ Forward pass successful")
    print(f"  Output shape: {output.shape}")
    print(f"  Vel probs shape: {vel_probs.shape}")
except Exception as e:
    print(f"✗ Forward pass failed: {e}")
    raise

# Loss & backward
try:
    criterion = torch.nn.MSELoss()
    loss = criterion(output, target_seq)
    print(f"✓ Loss computed: {loss.item():.6f}")
    
    loss.backward()
    print(f"✓ Backward pass successful")
except Exception as e:
    print(f"✗ Loss/backward failed: {e}")
    raise

# Check gradients
grad_ok = False
for name, p in model.named_parameters():
    if p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0:
        grad_ok = True
        print(f"✓ Gradient: {name} | norm={p.grad.norm().item():.6f}")
        break

print(f"\n{'='*50}")
print(f"✓✓ SANITY CHECK PASSED ✓✓" if grad_ok and loss.item() > 0 else "✗ Issues detected")
print(f"{'='*50}")

✓ Input shape: torch.Size([2, 10, 1, 28, 28]), Target shape: torch.Size([2, 10, 1, 28, 28])
✓ Forward pass successful
  Output shape: torch.Size([2, 10, 1, 28, 28])
  Vel probs shape: torch.Size([2, 20, 25])
✓ Loss computed: 0.001281
✓ Backward pass successful
✓ Gradient: cell.conv_h.weight | norm=0.000050

✓✓ SANITY CHECK PASSED ✓✓


In [13]:
# Training stability check: 5 epochs on synthetic data

print("\n" + "="*60)
print("TRAINING STABILITY CHECK (5 epochs)")
print("="*60)

# Create fresh model and data
model = Seq2SeqFERNN(
    input_channels=1,
    hidden_channels=16,
    height=28,
    width=28,
    v_range=2,
    decoder_conv_layers=1,
    pool_type='max'
)
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# Generate mini dataset (5 sequences with integer speeds)
torch.manual_seed(42)
train_seqs = [generate_moving_dot(batch_size=1, seq_len=20, speed=int(s)) for s in [0, 1, 1, 2, 0]]
input_seqs = torch.cat([s[:, :10] for s in train_seqs], dim=0)  # (5, 10, 1, 28, 28)
target_seqs = torch.cat([s[:, 10:] for s in train_seqs], dim=0)  # (5, 10, 1, 28, 28)

losses = []
for epoch in range(5):
    epoch_loss = 0
    
    # Mini-batch training
    for b in range(5):
        optimizer.zero_grad()
        
        output, _ = model(
            input_seqs[b:b+1],
            pred_len=10,
            teacher_forcing_ratio=0.5,
            target_seq=target_seqs[b:b+1],
            return_vel_probs=True
        )
        
        loss = criterion(output, target_seqs[b:b+1])
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / 5
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/5 | Loss: {avg_loss:.6f} | {'↓' if epoch > 0 and avg_loss < losses[epoch-1] else '↑'}")

# Check stability
print("\nStability Analysis:")
print(f"  Initial loss: {losses[0]:.6f}")
print(f"  Final loss:   {losses[-1]:.6f}")
print(f"  Trend:        {'STABLE ✓' if all(torch.isfinite(torch.tensor(l)) for l in losses) else 'NaN/Inf ✗'}")
print(f"  Decreasing:   {'Yes ✓' if losses[-1] < losses[0] else 'No (but may stabilize)'}")

if all(torch.isfinite(torch.tensor(l)) for l in losses) and losses[-1] < losses[0]:
    print("\n✓✓ TRAINING STABLE ✓✓")
elif all(torch.isfinite(torch.tensor(l)) for l in losses):
    print("\n✓ TRAINING STABLE (no divergence, loss may need more epochs to decrease)")
else:
    print("\n✗ TRAINING UNSTABLE (NaN/Inf detected)")


TRAINING STABILITY CHECK (5 epochs)
Epoch 1/5 | Loss: 0.001250 | ↑
Epoch 2/5 | Loss: 0.001125 | ↓
Epoch 3/5 | Loss: 0.000961 | ↓
Epoch 4/5 | Loss: 0.000898 | ↓
Epoch 5/5 | Loss: 0.000671 | ↓

Stability Analysis:
  Initial loss: 0.001250
  Final loss:   0.000671
  Trend:        STABLE ✓
  Decreasing:   Yes ✓

✓✓ TRAINING STABLE ✓✓
